# S&P 500 Derin Ogrenme ile Zaman Serisi Siniflandirma — v2 (Iyilestirilmis)

Bu notebook, S&P 500 sektor ETF'lerinin makro olaylar etrafindaki fiyat hareketlerini analiz ederek,
**T+5 gun sonra fiyatin yukselip yukselmeyecegini** tahmin eden bir **Ikili Siniflandirma (Binary Classification)** modeli kurar.

---
## v2'de Yapilan Degisiklikler

Onceki versiyonda **underfitting** (val_acc > train_acc) ve **platoda takilma** (loss ~0.686) sorunlari vardi.
Bu versiyonda asagidaki degisiklikler yapildi:

| Parametre | Eski | Yeni |
|---|---|---|
| Dropout orani | 0.5 | **0.2** |
| Dense katmanlar | 64 -> 32 | **128 -> 64 -> 32** |
| BatchNormalization | yok | **var** |
| Aktivasyon fonksiyonu | ReLU | **swish** |
| Learning rate | 0.0001 | **0.001** |
| Weight decay | 0.004 | **0.0001** |
| Batch size | 512 | **128** |
| Epoch | 750 | **200** |
| EarlyStopping patience | 75 | **15** |
| restore_best_weights | False | **True** |
| ReduceLROnPlateau | yok | **var** |

---
**Veri Seti:** `sp500_deep_learning_massive_data.csv` (~257.000 satir)
**Model:** Ileri Beslemeli Yapay Sinir Agi (Feedforward / Dense) + BatchNorm
**Framework:** TensorFlow / Keras

---
## Kutuphane Yuklemesi

In [ ]:
# ============================================================
# Temel Kutuphaneler
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Sklearn - On Isleme
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras - Derin Ogrenme
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization   # YENI: BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau           # YENI: ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.optimizers import AdamW

# Tekrarlanabilirlik icin rastgelelik tohumunu sabitle
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Surumu: {tf.__version__}")
print(f"NumPy Surumu: {np.__version__}")
print(f"Pandas Surumu: {pd.__version__}")
print("\nTum kutuphaneler basariyla yuklendi!")
print()

---
## Adim 1: Veriyi Okuma ve Hedef Degisken (Y) Olusturma

**Mantik:**
Her `[Olay_Ismi, Hisse]` grubu icinde, kapanis fiyatini 5 gun ileri kaydirarak (`shift(-5)`) T+5 gun sonraki fiyati buluyoruz.
- T+5 fiyati > Bugunku fiyat -> **Hedef = 1 (Yukselis)**
- T+5 fiyati <= Bugunku fiyat -> **Hedef = 0 (Dusus)**

In [ ]:
# ============================================================
# 1.1 - CSV Dosyasini Oku
# ============================================================
DOSYA_YOLU = "sp500_deep_learning_massive_data.csv"

df = pd.read_csv(DOSYA_YOLU)

print(f"Veri Seti Boyutu: {df.shape[0]:,} satir x {df.shape[1]} sutun")
print(f"\nSutunlar:\n{list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# 1.2 - Hedef Degisken (Y) Olusturma
# ============================================================
df = df.sort_values(by=['Olay_Ismi', 'Hisse', 'T0_Goreceli_Gun']).reset_index(drop=True)

df['Fiyat_T5'] = df.groupby(['Olay_Ismi', 'Hisse'])['Duzeltilmis_Kapanis'].shift(-5)

# Hedef degiskeni olustur: 1 = Yukselis, 0 = Dusus
df['Hedef'] = (df['Fiyat_T5'] > df['Duzeltilmis_Kapanis']).astype(int)

# T+5 hesaplanamayan (NaN) satirlari sil
satirlar_once = len(df)
df = df.dropna(subset=['Fiyat_T5']).reset_index(drop=True)
satirlar_sonra = len(df)

print(f"Silinen NaN satir sayisi: {satirlar_once - satirlar_sonra:,}")
print(f"Kalan satir sayisi: {satirlar_sonra:,}")
print(f"\nHedef Degisken Dagilimi:")
print(df['Hedef'].value_counts())
print(f"\nYukselis Orani: %{df['Hedef'].mean()*100:.1f}")

---
## Adim 2: Ozellik Secimi ve 2D Veri Hazirligi

**Yaklasim:**
Kayan pencere (sliding window) veya 3 boyutlu veri hazirligi **KULLANILMIYOR**.
Veri dogrudan `(Orneklem Sayisi, Ozellik Sayisi)` boyutunda standart 2D olarak hazirlaniyor.
Her satir bagimsiz bir orneklem olarak modele veriliyor.

In [ ]:
# ============================================================
# 2.1 - Ozellik Sutunlarini Sec
# ============================================================
OZELLIK_SUTUNLARI = [
    'Log_Getiri',
    'Volatilite_10g',
    'Volatilite_30g',
    'RSI_14',
    'MACD_12_26_9',
    'MACDh_12_26_9',
    'MACDs_12_26_9',
    'BBL_20_2.0',
    'BBU_20_2.0'
]

print(f"Kullanilacak Ozellik Sayisi: {len(OZELLIK_SUTUNLARI)}")
print(f"Ozellikler: {OZELLIK_SUTUNLARI}")

# Eksik deger kontrolu
eksik = df[OZELLIK_SUTUNLARI].isnull().sum()
print(f"\nOzelliklerdeki Eksik Degerler:\n{eksik[eksik > 0]}")

# Eksik degerleri olan satirlari temizle
df = df.dropna(subset=OZELLIK_SUTUNLARI).reset_index(drop=True)
print(f"\nTemizleme sonrasi kalan satir: {len(df):,}")

In [ ]:
# ============================================================
# 2.2 - 2D Veri Hazirligi (Orneklem, Ozellik)
# ============================================================
# Kayan pencere veya 3D donusum YOK.
# Her satir dogrudan bir orneklem olarak kullaniliyor.

X = df[OZELLIK_SUTUNLARI].values
y = df['Hedef'].values

ozellik_sayisi = X.shape[1]

print(f"X boyutu: {X.shape}  -> (Orneklem, Ozellik)")
print(f"y boyutu: {y.shape}  -> (Orneklem,)")
print(f"\nToplam orneklem sayisi: {X.shape[0]:,}")
print(f"Ozellik sayisi: {ozellik_sayisi}")

---
## Adim 3: Train/Test Ayrimi ve Olceklendirme (Leakage-Safe)

**Veri Sizintisini (Data Leakage) Onleme:**
- `train_test_split` ile `shuffle=False` kullaniliyor (zaman serisi sirasi korunuyor)
- `StandardScaler` sadece egitim setine `fit` ediliyor
- Ardindan hem egitim hem test setine `transform` uygulaniyor

In [ ]:
# ============================================================
# 3.1 - Train / Test Bolmesi (shuffle=False) + Leakage-safe Scaling
# ============================================================
# Zaman serisi oldugu icin shuffle YAPILMIYOR!
# Gecmis veri ile egitip, gelecek veri ile test ediyoruz.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)

print(f"Bolme oncesi X boyutu: {X.shape}")
print(f"\nEgitim Seti:  X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test Seti:    X_test={X_test.shape},  y_test={y_test.shape}")

# --- Leakage-safe olceklendirme ---
# StandardScaler SADECE egitim setine fit ediliyor
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit + transform (sadece train)
X_test  = scaler.transform(X_test)        # sadece transform (test)

print(f"\nEgitim Setindeki Yukselis Orani: %{y_train.mean()*100:.1f}")
print(f"Test Setindeki Yukselis Orani:   %{y_test.mean()*100:.1f}")
print(f"\nScaler fit edildi (sadece train): mean={scaler.mean_.round(3)}")

---
## Adim 4: Feedforward (Dense) Model Mimarisi — v2 (Daha Derin + BatchNorm)

**Yeni Mimari:**
```
Input (9,) 
  -> Dense (128, swish) -> BatchNorm -> Dropout (0.2)
  -> Dense (64, swish)  -> BatchNorm -> Dropout (0.2)
  -> Dense (32, swish)  -> Dropout (0.2)
  -> Dense (1, sigmoid)
```

**Degisikliklerin Sebepleri:**
- **Dense(128 -> 64 -> 32):** Bir katman daha eklendi; 9 ozelligin zengin temsilini ogrenmek icin kapasite artirildi.
- **BatchNormalization:** Finansal veride gradient stabilitesini saglar, egitimi hizlandirir ve platoya takilmayi azaltir.
- **swish aktivasyonu:** `f(x) = x * sigmoid(x)`. ReLU'nun "dying ReLU" sorununu yasamaz, kucuk sinyalli finansal veride daha yumusak gradient verir.
- **Dropout(0.2):** 0.5 cok agresif oldugu icin underfitting yasaniyordu (val_acc > train_acc). 0.2 ile model daha iyi ogrenebilecek ama yine de overfitting'e karsi korumali olacak.

In [ ]:
# ============================================================
# 4.1 - Modeli Kur (v2 Mimarisi)
# ============================================================
model = Sequential([
    Input(shape=(ozellik_sayisi,)),

    # 1. Gizli Katman: 128 noron, swish aktivasyonu
    Dense(128, activation='swish'),
    BatchNormalization(),
    Dropout(0.2),

    # 2. Gizli Katman: 64 noron, swish aktivasyonu
    Dense(64, activation='swish'),
    BatchNormalization(),
    Dropout(0.2),

    # 3. Gizli Katman: 32 noron, swish aktivasyonu
    Dense(32, activation='swish'),
    Dropout(0.2),

    # Cikti Katmani: 1 noron, Sigmoid aktivasyonu
    # 0-1 arasi olasilik uretir (Ikili siniflandirma)
    Dense(1, activation='sigmoid')
])

# Model ozetini goster
model.summary()

---
## Adim 5: Modeli Derleme ve Egitme (Compile & Fit) — v2

**Derleme Ayarlari:**
- `loss='binary_crossentropy'`: Ikili siniflandirma icin standart kayip fonksiyonu
- `optimizer=AdamW(lr=0.001, weight_decay=0.0001)`: Learning rate 10 kat artirildi, weight decay azaltildi
- `metrics=['accuracy']`: Dogruluk metrigi

**Egitim Ayarlari:**
- `batch_size=128`: 512 cok yumusatiyordu, 128 zaman serisinde daha iyi gradient sinyali verir
- `epochs=200`: 750 gereksizdi, patience'in dogru kaliplamasi icin yeterli
- `EarlyStopping(patience=15, restore_best_weights=True)`: En onemli yenilik — **restore_best_weights** ile egitim sonunda en iyi epoch'un agirliklari geri yukleniyor. Eski modelde EarlyStopping bitince son (kotulesen) agirliklar kaliyordu.
- `ReduceLROnPlateau`: Val loss 5 epoch iyilesmezse learning rate'i yariya dusurur. Bu, platoda sikisan modellerin "kirilmasi" icin cok etkilidir.

In [ ]:
# ============================================================
# 5.1 - Modeli Derle (Compile)
# ============================================================
# YENI: Learning rate 10 kat artirildi (0.0001 -> 0.001)
# YENI: Weight decay azaltildi (0.004 -> 0.0001)
ozel_optimizer = AdamW(learning_rate=0.001, weight_decay=0.0001)

model.compile(
    loss='binary_crossentropy',       # Ikili siniflandirma kaybi
    optimizer=ozel_optimizer,          # AdamW (iyilestirilmis ayarlar)
    metrics=['accuracy']               # Dogruluk metrigi
)

print("Model derlendi.")
print(f"   Loss: binary_crossentropy")
print(f"   Optimizer: AdamW(learning_rate=0.001, weight_decay=0.0001)")
print(f"   Metrik: Accuracy")

In [ ]:
# ============================================================
# 5.2 - Callback'leri Tanimla
# ============================================================
# YENI: restore_best_weights=True ile egitim sonunda en iyi agirliklar geri yuklenir
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,    # KRITIK: En iyi epoch'un agirliklarini geri yukler
    verbose=1
)

# YENI: Val loss platoya girince learning rate'i otomatik azaltir
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        # LR'yi yariya dusur
    patience=5,        # 5 epoch iyilesme yoksa tetiklen
    min_lr=1e-6,       # LR'nin inebilecegi en dusuk seviye
    verbose=1
)

print("Callback'ler hazirlandi:")
print("   EarlyStopping(patience=15, restore_best_weights=True)")
print("   ReduceLROnPlateau(factor=0.5, patience=5)")

In [ ]:
# ============================================================
# 5.3 - Modeli Egit (Fit)
# ============================================================
print("Model egitimi basliyor...\n")

gecmis = model.fit(
    X_train, y_train,
    validation_split=0.2,              # Egitim setinin %20'si dogrulama icin ayrilir
    epochs=200,                        # YENI: 750 -> 200 (gereksizdi)
    batch_size=128,                    # YENI: 512 -> 128
    callbacks=[early_stop, lr_scheduler],
    verbose=1
)

print("\nEgitim tamamlandi!")

---
## Adim 6: Model Performansini Gorsellestir ve Degerlendir

In [ ]:
# ============================================================
# 6.1 - Egitim Surecini Gorsellestir (Loss & Accuracy Grafikleri)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Sol Grafik: Loss (Kayip) ---
axes[0].plot(gecmis.history['loss'], label='Egitim Kaybi (Train Loss)', linewidth=2)
axes[0].plot(gecmis.history['val_loss'], label='Dogrulama Kaybi (Val Loss)', linewidth=2, linestyle='--')
axes[0].set_title('Kayip Fonksiyonu (Loss) - Epoch Bazinda', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Sag Grafik: Accuracy (Dogruluk) ---
axes[1].plot(gecmis.history['accuracy'], label='Egitim Dogrulugu (Train Acc)', linewidth=2)
axes[1].plot(gecmis.history['val_accuracy'], label='Dogrulama Dogrulugu (Val Acc)', linewidth=2, linestyle='--')
axes[1].set_title('Dogruluk (Accuracy) - Epoch Bazinda', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('egitim_grafikleri_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Overfitting tespiti icin bilgi notu
son_train_loss = gecmis.history['loss'][-1]
son_val_loss = gecmis.history['val_loss'][-1]
fark = abs(son_train_loss - son_val_loss)
print(f"\nSon Epoch Istatistikleri:")
print(f"   Train Loss: {son_train_loss:.4f}")
print(f"   Val Loss:   {son_val_loss:.4f}")
print(f"   Fark:       {fark:.4f}")
if fark > 0.1:
    print("   Egitim ve dogrulama kaybi arasinda belirgin fark var -> Overfitting riski!")
else:
    print("   Egitim ve dogrulama kaybi yakin -> Model dengeli ogrenmis.")

In [ ]:
# ============================================================
# 6.2 - Test Seti Uzerinde Nihai Degerlendirme
# ============================================================

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print("=" * 60)
print("MODEL DEGERLENDIRME OZETI (v2)")
print("=" * 60)
print(f"\n   Test Kaybi (Loss):     {test_loss:.4f}")
print(f"   Test Dogrulugu (Acc):  %{test_acc*100:.2f}")
print(f"\n   Egitim Epoch Sayisi:   {len(gecmis.history['loss'])}")
print(f"   Ozellik Sayisi:        {ozellik_sayisi}")
print(f"   Toplam Parametre:      {model.count_params():,}")
print(f"   Mimari:                Dense(128)->BN->Drop(0.2) -> Dense(64)->BN->Drop(0.2) -> Dense(32)->Drop(0.2) -> Dense(1)")
print(f"   Aktivasyonlar:         swish (gizli) / Sigmoid (cikti)")
print(f"   Regularizasyon:        Dropout(0.2) + BatchNorm + EarlyStopping(patience=15, restore_best=True) + ReduceLROnPlateau")
print(f"   Optimizer:             AdamW(lr=0.001, weight_decay=0.0001)")
print(f"   Batch Size:            128")
print("\n" + "=" * 60)
print("Analiz tamamlandi!")
print("=" * 60)

---
## Adim 7: v1 ile v2 Karsilastirma Notlari

### v1 Sonuclari (referans)
- Test Loss: ~0.6867
- Test Acc: ~%55.77 — %55.93
- Sorun: Val acc > Train acc (underfitting), loss platoda sikismis (0.686)

### v2 Beklentisi
- **Test Loss: 0.678 — 0.682** (hedef)
- **Test Acc: %57 — %59** (hedef)
- Train ve Val acc birbirine yakin olmali (Dropout 0.2 ile dengeli)

### Gozlem Notlari
Egitim sirasinda ReduceLROnPlateau ciktisini takip et:
- **"ReduceLROnPlateau reducing learning rate to ..."** mesajlari gorursen, model platoya takildiginda LR'nin yarilandigi anlamina gelir ve genelde bunun sonrasinda val_loss bir kac puan daha duser.
- **EarlyStopping erken tetiklenirse** (orn. 25-30. epoch'ta), bu normal — patience=15 ile en iyi noktayi korur ve `restore_best_weights=True` sayesinde dogru agirliklar geri yuklenir.

### Eger hala %56'da takiliysa
Bu durumda sorun modelde degil veride. 9 ozellik T+5 fiyat yonunu tahmin icin yetersiz olabilir. Ek ozellik muhendisligi (daha uzun donem teknik gostergeler, sektor rotasyon sinyalleri, VIX vb.) gerekebilir.